# llm-memlab Colab Quickstart

Notebook นี้แบ่งเป็น 5 ส่วนชัด ๆ: **Setup** ติดตั้ง lib, **Load Model** โหลดโมเดลจาก Hugging Face, **Debug Runtime** ตรวจ backend/GPU, **Optimize/Benchmark** รัน memory-first + serving benchmark, และ **View Report** เปิด HTML dashboard.

## 1. Setup

Clone project และติดตั้งแบบ editable เพื่อให้ Colab เรียก `llm_memlab` ได้ทันที.

In [ ]:
!git clone https://github.com/thhanabun/LLMOptimization.git
%cd LLMOptimization
!python -m pip install -e .[torch,transformers]

## 2. Configure Paths

ตั้ง path กลางผ่าน environment variables เพื่อไม่ hardcode และให้ใช้ซ้ำกับ local/Kaggle/cloud GPU ได้.

In [ ]:
import os
os.environ.setdefault("LLM_MEMLAB_MODEL_ROOT", "/content/hf_models")
os.environ.setdefault("LLM_MEMLAB_OUT_DIR", "/content/llm_memlab_outputs")
os.environ.setdefault("LLM_MEMLAB_DEVICE", "auto")
os.environ.setdefault("LLM_MEMLAB_DTYPE", "auto")

## 3. Load Model From Hugging Face

โหลด model ลง local Colab runtime ด้วย `snapshot_download` แล้วตั้ง `LLM_MEMLAB_MODEL` ให้ lib ใช้ต่อได้ทันที.

- ค่า default เป็น tiny random Llama เพื่อ smoke test เร็ว ๆ
- เปลี่ยน `HF_MODEL_ID` เป็น `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, `Qwen/Qwen3-1.7B` หรือ model อื่นได้ถ้า runtime รับไหว
- ถ้าเป็น gated/private model ให้ตั้ง `HF_TOKEN` ก่อน

In [ ]:
from huggingface_hub import snapshot_download

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "hf-internal-testing/tiny-random-LlamaForCausalLM")
HF_TOKEN = os.environ.get("HF_TOKEN")
local_dir = os.path.join(os.environ["LLM_MEMLAB_MODEL_ROOT"], HF_MODEL_ID.replace("/", "__"))

model_path = snapshot_download(repo_id=HF_MODEL_ID, local_dir=local_dir, token=HF_TOKEN)
os.environ["LLM_MEMLAB_MODEL"] = model_path
print("Downloaded model to:", model_path)

## 4. Debug Runtime

เช็กก่อนว่า runtime มี Torch/CUDA/Triton/vLLM อะไรพร้อมบ้าง และดู estimate memory คร่าว ๆ. ส่วนนี้ช่วยบอกว่า backend ไหนจะ fallback.

In [ ]:
!python -m llm_memlab backend-demo
!python -m llm_memlab estimate --preset 7b-like --seq 2048 --batch 1 --training inference

## 5. Optimize / Benchmark

รัน workflow หลัก: `memory-first-hf-bench`, `serving-bench`, และ export benchmark dashboard. ถ้า vLLM ไม่พร้อม จะเห็น fallback reason ชัดเจนใน report.

In [ ]:
!python examples/cloud_smoke.py --tokens 1

## 6. View Report

เปิด HTML dashboard ใน notebook เพื่อดู latency, quality, memory, backend และ fallback reason.

In [ ]:
from IPython.display import HTML, display
dashboard = os.path.join(os.environ["LLM_MEMLAB_OUT_DIR"], "cloud_dashboard.html")
if os.path.exists(dashboard):
    display(HTML(open(dashboard, encoding="utf-8").read()))